# 02 · Factorial de dos factores — ANOVA de dos vías (Python)

**Semana 2 — Bloqueo y factoriales.**

## Objetivos
- Ajustar un factorial de dos factores con interacción.
- Interpretar la **gráfica de interacción** y aplicar la estrategia correcta.

> Teoría: [`../teoria/02-factoriales-interacciones.md`](../teoria/02-factoriales-interacciones.md) y [`../teoria/03-anova-dos-vias.md`](../teoria/03-anova-dos-vias.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.graphics.factorplots import interaction_plot

np.random.seed(2)

## 1. Los datos

Vida (horas) de una batería según el **tipo de material** (3) y la **temperatura** (15, 70, 125 °F), con $n=4$ réplicas por celda (Montgomery, ej. 5.3).

In [ ]:
df = pd.read_csv('../datos/vida-bateria.csv')
df['material'] = df['material'].astype('category')
df['temperatura'] = df['temperatura'].astype('category')
df.groupby(['material','temperatura'], observed=True)['vida'].mean().unstack()

## 2. Gráfica de interacción

Primero la herramienta visual clave: ¿son paralelas las líneas?

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
interaction_plot(df['temperatura'].astype(int), df['material'], df['vida'],
                 ax=ax, ms=8)
ax.set_xlabel('Temperatura (°F)'); ax.set_ylabel('Vida media (h)')
ax.set_title('Gráfica de interacción material × temperatura'); plt.show()

Las líneas **no** son paralelas (el material 3 aguanta mejor las temperaturas altas): esto anticipa una **interacción**.

## 3. ANOVA de dos vías con interacción

Modelo $y_{ijk}=\mu+\tau_i+\beta_j+(\tau\beta)_{ij}+\varepsilon_{ijk}$.

In [ ]:
modelo = ols('vida ~ C(material)*C(temperatura)', data=df).fit()
tabla = sm.stats.anova_lm(modelo, typ=2)
print(tabla.round(4))

**Estrategia: mirar primero la interacción.** El término `C(material):C(temperatura)` es significativo ($p\approx0.019$): material y temperatura **actúan conjuntamente**. Por tanto, los efectos principales se interpretan con cautela; hay que analizar las **celdas**.

## 4. Verificación de supuestos

In [ ]:
resid = modelo.resid; ajust = modelo.fittedvalues
fig, ax = plt.subplots(1, 2, figsize=(11,4))
sm.qqplot(resid, line='s', ax=ax[0]); ax[0].set_title('Q-Q de residuales')
ax[1].scatter(ajust, resid); ax[1].axhline(0, color='gray', ls='--')
ax[1].set_xlabel('Ajustados'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuales vs. ajustados')
plt.tight_layout(); plt.show()
print('Shapiro:', stats.shapiro(resid))

## 5. Conclusiones
- **Interacción significativa** material × temperatura ($p\approx0.019$): la elección del material depende de la temperatura de operación.
- A baja temperatura los materiales rinden parecido; a alta temperatura el **material 3** es claramente superior.
- Recomendación: si el equipo opera caliente, usar material 3.

> Equivalente en R: [`02-factorial-dos-vias_r.ipynb`](02-factorial-dos-vias_r.ipynb).